# (노트북 15 · Run All) VoD 통합 비교 — **FMCW 레이더 화면 기준 파이프라인**

발표·웹에서 쓰는 **다중 센서 파이프라인**(SAR 광역 → UAV 추적·EO/IR → **FMCW·근거리** → 드론 EO/IR 근접 식별)을 한 줄로 이었을 때, **운용자가 실제로 손이 가는 곳은 후반의 FMCW 콘솔**입니다.

이 노트북은 VoD **오프라인 데이터**로 그 **“레이더 화면 한 스냅샷”**을 가정하고, **위에서 아래로 Run All** 하면서:

1. **그 화면에 올라오는 레이어**(원시/군집, BEV 히트맵, 식별 패널)가 **무엇에서 오는지**  
2. **앞단(SAR·UAV·드론)이 이 스냅샷에 어떻게 녹아 있는지** (데이터상으로는 카메라·YOLO·동기 프레임으로 대응)  
3. **LiDAR·GT는 오직 B구간 검증·상한**이라는 점  

을 **순서대로, 꽤 자세히** 짚도록 재구성했습니다.

> `13_radar_bev_lidar_yolo_compare.ipynb`와 **실행 코드는 동일**합니다. 차이는 **설명을 FMCW 운용석 관점으로 통일**한 것입니다.

---

## A. 최종 운용 가정 (FMCW 단독) — 요약

**실제 배치에서는 FMCW 레이더 스트림이 메인 입력**입니다. 내부적으로는 대략:

1. **전처리** (필터·좌표 정합)  
2. **군집화** (DBSCAN 등) → 후보  
3. **후보 선택·추적** (연속 프레임)  
4. **(선택) BEV CNN** 히트맵·피크  
5. **위험도·등급** — `high_confidence` / `medium_confidence` / `low_confidence` 처럼 **레이더 특징 기반** 권장 (점 수, 속도 일관성, 추적 지속성 등)

**LiDAR는 이 노트북에서만** “같은 프레임의 정답·규칙 기반 상한”으로 등장합니다. **운영 UI의 핵심 구분으로 LiDAR 검증 태그를 쓰지 않는 것**이 발표와 맞습니다.

---

## B. 전체 파이프라인 vs 이 노트북의 대응

| 파이프 단계 | 역할 | 이 노트북에서의 대응 |
|-------------|------|----------------------|
| SAR 광역 | AOI·변화 | 직접 파일 없음 → **이미 정해진 `FRAME_ID` 한 틱**으로 진입했다고 가정 |
| UAV·EO/IR | 연속 추적·영상 | **원본 카메라** + **YOLO 패널**이 같은 시간축 **EO/IR 슬롯** |
| **FMCW** | ≤15 km 근거리 | **레이더 `.bin`**, **융합 API의 radarDetections**, **레이더 BEV CNN**, **BEV 피크·레이더 XY** |
| 드론 EO/IR | 근접 확정 | 데모에선 **카메라/영상**으로 대체 표현 |

---

## C. “FMCW 화면”에 쌓이는 레이어 (실행 후 2×4로 한 장 요약)

- **식별 패널(이미지)**: YOLO 박스 — 레이더 후보와 **나란히 보는** 교차 모달리티.  
- **BEV**: 레이더만 학습 CNN 피크 vs (참고) LiDAR 학습 CNN·GT·DBSCAN.  
- **레이더 X 표시**: 융합 API centroid 우선, 없으면 로컬 DBSCAN 중심.

---

## D. 아래 셀 진행 순서 (FMCW 시점)

| 순서 | 구간 | FMCW 화면 관점 |
|------|------|----------------|
| 0 | import | 공통 도구·좌표계 |
| 1 | 설정 | **동기 프레임 ID = 레이더 한 틱** |
| 2 | 카메라 | **EO/IR 참조 뷰** |
| 3 | 융합 API | **서버측** 군집·YOLO·(실험) LiDAR |
| 4 | YOLO 그림 | **식별 오버레이** |
| 5 | 로컬 로드 | **온디바이스** RAW·캘리브·(검증) LiDAR·라벨 |
| 6 | BEV CNN 학습 | **히트맵 헤드** (배포 시엔 사전학습) |
| 7 | 추론 | 현재 틱 **피크·레이더XY·DBSCAN** |
| 8 | 2×4 | **통합 디스플레이** |

**필요**: `vod-received/view_of_delft_PUBLIC`(또는 `VOD_ROOT`), `torch`, `numpy`, `matplotlib`, `sklearn`, `requests`, `PIL` · (선택) `ai-inference`  
**모듈**: `bev_lidar_detector_train.py`, `vod_compare_utils.py`

---

## 발표용 문장 (복사)

- *「최종 운용은 FMCW 레이더 단독이며, LiDAR는 데이터셋 기반 오프라인 검증·상한 비교용입니다.»*  
- *「노트북은 레이더 콘솔에 올라오는 레이어를 한 프레임에서 순서대로 채워 넣는 흐름으로 설명합니다.»*



## 0. 공통 import · FMCW 준비



In [ ]:
from __future__ import annotations

import io
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
import numpy as np
import torch
from PIL import Image
from sklearn.cluster import DBSCAN

NOTEBOOK_DIR = Path.cwd().resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

import bev_lidar_detector_train as bev
import vod_compare_utils as vcu

# Windows 한글 (선택)
try:
    from matplotlib import font_manager, rc
    _fp = Path("C:/Windows/Fonts/malgun.ttf")
    if _fp.is_file():
        _fn = font_manager.FontProperties(fname=str(_fp)).get_name()
        rc("font", family=_fn)
        rc("axes", unicode_minus=False)
except Exception:
    pass

print("ready:", NOTEBOOK_DIR)


---
### FMCW 화면 · 준비 0/7 — 공통 도구

아래부터 그리는 그림은 전부 **차량 기준 좌표(velodyne 계열)·BEV 그리드** 위입니다.  
실제 FMCW UI도 **동일한 외부·캘리브레이션**을 전제로 레이더 점·트랙·히트맵을 겹칩니다.


## 1. 설정: 데이터 루트 · 프레임 ID · AI 서버 URL


In [ ]:
DEFAULT_ROOT = NOTEBOOK_DIR / "vod-received" / "view_of_delft_PUBLIC"
DATASET_ROOT = Path(os.environ.get("VOD_ROOT", str(DEFAULT_ROOT))).resolve()
AI_INFERENCE_URL = os.environ.get("AI_INFERENCE_URL", "http://127.0.0.1:8001")

# ??? ??? (?? stem). ?: "01201"
FRAME_ID = os.environ.get("VOD_COMPARE_FRAME", "01201")
RADAR_VIEW_MODE = os.environ.get("VOD_COMPARE_RADAR_MODE", "5-scan")

lidar_dir = DATASET_ROOT / "lidar" / "training" / "velodyne"
radar_dir = bev.choose_radar_dir(DATASET_ROOT, RADAR_VIEW_MODE)
image_dir = DATASET_ROOT / "lidar" / "training" / "image_2"
label_dir = DATASET_ROOT / "lidar" / "training" / "label_2"
calib_dir = DATASET_ROOT / "lidar" / "training" / "calib"
if not calib_dir.is_dir():
    calib_dir = DATASET_ROOT / "calib"

paths = {
    "lidar": lidar_dir / f"{FRAME_ID}.bin",
    "radar": radar_dir / f"{FRAME_ID}.bin",
    "image": image_dir / f"{FRAME_ID}.jpg",
    "label": label_dir / f"{FRAME_ID}.txt",
    "calib": calib_dir / f"{FRAME_ID}.txt",
}

print("RADAR_VIEW_MODE:", RADAR_VIEW_MODE)
print("radar_dir:", radar_dir)
for k, pth in paths.items():
    print(k, "OK" if pth.is_file() else "MISSING", pth)

if not paths["lidar"].is_file():
    raise FileNotFoundError("LiDAR ?? ? FRAME_ID ?? VOD_ROOT ??")


---
### FMCW 화면 · 단계 1/7 — **동기 프레임 = 레이더 한 틱**

`FRAME_ID`(예: `01201`)는 **SAR/UAV가 이미 의심 구역을 좁힌 뒤**, FMCW가 **연속 관측 중인 그 순간의 샘플**이라고 보면 됩니다.

- `radar/.../velodyne/{FRAME_ID}.bin` → **이 콘솔의 메인 입력**  
- `lidar`·`label_2`·`calib` → **B 구간: 정답·LiDAR CNN 상한·DBSCAN 참고**  
- `image_2` → **UAV/드론 EO/IR와 같은 “광학 슬롯”** (시간 동기 stem)



## 2. 원본 카메라 이미지 보기


In [ ]:
if paths["image"].is_file():
    img_raw = Image.open(paths["image"]).convert("RGB")
    plt.figure(figsize=(10, 5))
    plt.imshow(img_raw)
    plt.title(f"원본 카메라 — frame {FRAME_ID}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    img_raw = None
    print("이미지 파일 없음")


---
### FMCW 화면 · 단계 2/7 — **EO/IR 참조 뷰 (UAV·드론 단계의 축)**

광역 SAR는 이 노트북에 파일로 없지만, **그 다음 UAV·드론이 내려주는 영상**은 운용석에서 **레이더 트랙 옆 패널**로 붙습니다.  
여기서는 **KITTI/VoD 카메라 한 장**이 그 역할을 합니다 — **BEV와 직접 픽셀 정합은 하지 않고**, “같은 프레임에서 무엇이 보이는가” **정성 참조**용입니다.



## 3. AI 서버: 레이더 DBSCAN + YOLO + (데이터셋) LiDAR 융합 (`/infer/vod/radar-fusion`)

**위치: B (오프라인·데모 비교)** — 최종 FMCW-only 배치 파이프라인에 LiDAR 입력을 넣는다고 가정하지 않습니다.

서버 응답에 LiDAR 교차검증이 포함될 수 있으면, 그것은 **후보의 공간적 타당성을 보는 실험적 단계**로 해석하면 됩니다. 서버가 꺼져 있으면 `FUSION = None`이고 이후는 로컬 레이더·LiDAR 파일로만 B 실험을 진행합니다.


In [ ]:
FUSION: dict | None = None
FUSION_ERROR: str | None = None

if paths["radar"].is_file():
    try:
        prev_id = str(max(0, int(FRAME_ID) - 1)).zfill(len(FRAME_ID))
        prev_radar = radar_dir / f"{prev_id}.bin"
        if not prev_radar.is_file():
            prev_radar = None
        FUSION = vcu.post_vod_radar_fusion(
            AI_INFERENCE_URL,
            paths["radar"],
            paths["image"] if paths["image"].is_file() else None,
            paths["lidar"],
            radar_prev_path=prev_radar,
        )
        print("fusion ok:", FUSION.get("ok"), "inferMs:", FUSION.get("inferMs"))
        print("radarDetections:", len(FUSION.get("radarDetections") or []))
        print("yoloDetections:", len(FUSION.get("yoloDetections") or []))
    except Exception as e:
        FUSION_ERROR = str(e)
        print("AI 융합 스킵:", FUSION_ERROR)
else:
    print("레이더 파일 없음 — 융합 생략")


---
### FMCW 화면 · 단계 3/7 — **서버 추론 슬롯 (`/infer/vod/radar-fusion`)**

운영 아키텍처에서는 이 부분이 **엣지/백엔드 추론 서비스**에 해당합니다.

- **레이더 RAW**(필요 시 이전 프레임) → **DBSCAN 등 군집** → `radarDetections` centroid  
- **같은 stem 이미지** → **YOLO** (식별 패널)  
- 요청에 **LiDAR가 실리면** 응답에 교차검증이 섞일 수 있음 → **B(실험)** 로만 해석

서버가 꺼져 있으면 `FUSION = None`이고, 이후는 **로컬 레이더·파일만**으로 FMCW 실험을 계속합니다.



### 3-1. YOLO 결과 이미지 (서버가 그린 박스)


In [ ]:
if FUSION and FUSION.get("annotatedImageBase64"):
    yolo_vis = vcu.b64_to_pil_image(FUSION["annotatedImageBase64"])
    plt.figure(figsize=(10, 5))
    plt.imshow(yolo_vis)
    plt.title("YOLO 검출 (ai-inference)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
elif paths["image"].is_file():
    try:
        solo = vcu.post_yolo_image_only(AI_INFERENCE_URL, paths["image"])
        if solo.get("annotatedImageBase64"):
            plt.figure(figsize=(10, 5))
            plt.imshow(vcu.b64_to_pil_image(solo["annotatedImageBase64"]))
            plt.title("YOLO only (/infer/image)")
            plt.axis("off")
            plt.show()
    except Exception as e:
        print("YOLO 단독 호출 실패:", e)
else:
    print("표시할 YOLO 이미지 없음")


---
### FMCW 화면 · 단계 4/7 — **식별 오버레이 (YOLO)**

레이더 BEV에서 후보가 떠도, 운용자는 **“무엇인지”**를 EO/YOLO 쪽에서 확인합니다.  
이 셀은 **융합 API가 그린 박스** 또는 **이미지 단독 YOLO**를 띄웁니다 — **FMCW 화면의 오른쪽/아래 “영상 패널”**에 대응합니다.



## 4. LiDAR / 레이더 / 캘리브 / 라벨 로드 (로컬) — **오프라인 검증용 데이터**

VoD는 **레이더·LiDAR·카메라·라벨**이 같이 있어, **같은 프레임**에서 A축(레이더 후보)과 **GT/LiDAR 규칙**을 나란히 놓을 수 있습니다.  
**최종 시스템**에서는 이 LiDAR 스트림이 없다고 보고, 여기서 나온 비교는 **B: 보조 검증·한계 파악·튜닝 참고**로만 쓰면 됩니다.


In [ ]:
lidar_pts = bev.parse_lidar_bin(paths["lidar"])
calib = bev.parse_calib_txt(paths["calib"])
label_rows = bev.parse_kitti_label(paths["label"]) if paths["label"].is_file() else []
radar_pts = np.fromfile(paths["radar"], dtype=np.float32).reshape(-1, 7) if paths["radar"].is_file() else np.zeros((0, 7))

print("LiDAR points:", lidar_pts.shape[0], "| Radar points:", radar_pts.shape[0], "| label rows:", len(label_rows))
print("calib:", calib is not None)

bev_cfg = bev.BevGridConfig()
gt_footprints = vcu.label_rows_to_velo_footprints(label_rows, calib) if calib else []


---
### FMCW 화면 · 단계 5/7 — **온디바이스 적재 (RAW → 텐서 준비)**

지금까지는 API가 섞일 수 있는 **서버 경로**였고, 여기서는 **노트북이 직접** 같은 프레임을 읽습니다.

- `radar_pts` → **FMCW 점군** (BEV 투영·DBSCAN의 입력)  
- `lidar_pts` + `label_rows` + `calib` → **오프라인에서만** GT 다각형·LiDAR BEV CNN·DBSCAN 박스 생성

즉 **운용 FMCW 단말**이라면 `lidar_pts` 분기는 없고, **나머지 파이프는 동일한 사고방식**으로 확장하면 됩니다.



## 5. Radar-only BEV CNN ?? + LiDAR ?? ???? ? **???? ?? ??**

??? ?? **????** ????, **LiDAR? validation???** ?? ??? ???.  
?, **?? ??? radar-only**, **??/?? ??? LiDAR-assisted** ???.

?? ?? ?? 3? ??? ? ?? ????.

- **baseline 3-scan + Tiny CNN**: `log_count`, `max_abs_vr_comp`
- **rich 5-scan + Residual CNN**: RCS / ?? / ?? ?? 8??
- **temporal 5-scan + U-Net**: ????? ??? 12??

**?? ?? ??**  
`0.5 * label_2 micro-F1 + 0.3 * LiDAR cluster F1 + 0.2 * LiDAR support ratio`

**?? ??**  
`RADAR_BENCH_MAX_TRAIN`(240), `RADAR_BENCH_MAX_VAL`(120), `RADAR_BENCH_EPOCHS`(8), `RADAR_BENCH_BATCH`(4), `RADAR_BENCH_LR`(8e-4), `BEV_WEIGHT_DECAY`(0.02)


In [ ]:
radar_specs = bev.default_radar_experiment_specs()
max_train = int(os.environ.get("RADAR_BENCH_MAX_TRAIN", os.environ.get("BEV_QUICK_MAX_TRAIN", "240")))
max_val = int(os.environ.get("RADAR_BENCH_MAX_VAL", os.environ.get("BEV_QUICK_MAX_VAL", "120")))
epochs = int(os.environ.get("RADAR_BENCH_EPOCHS", os.environ.get("BEV_QUICK_EPOCHS", "8")))
batch = int(os.environ.get("RADAR_BENCH_BATCH", os.environ.get("BEV_QUICK_BATCH", "4")))
lr = float(os.environ.get("RADAR_BENCH_LR", os.environ.get("BEV_LR", "8e-4")))
wd = float(os.environ.get("BEV_WEIGHT_DECAY", "0.02"))

radar_benchmark = bev.run_radar_only_benchmark(
    DATASET_ROOT,
    bev_cfg=bev_cfg,
    experiments=radar_specs,
    train_split="train",
    val_split="val",
    max_train=max_train,
    max_val=max_val,
    epochs=epochs,
    batch_size=batch,
    lr=lr,
    weight_decay=wd,
    num_workers=0,
)

leaderboard = radar_benchmark["leaderboard"]
radar_experiments = radar_benchmark["experiments"]
best_radar = radar_benchmark["best"]
device = torch.device(radar_benchmark["device"])
class_names = ["Vehicle", "Pedestrian", "Cyclist"]

print("RADAR benchmark ready | device:", device)
for row in leaderboard:
    print(
        f"#{row['rank']} {row['name']} | mode={row['radar_mode']} | model={row['model_name']} | "
        f"val_bce={row['best_val_bce']:.5f} | gt_f1={row['micro_f1']:.4f} | "
        f"lidar_f1={row['lidar_cluster_f1']:.4f} | support={row['lidar_support_ratio']:.4f} | "
        f"score={row['selection_score']:.4f}"
    )

if best_radar is None:
    raise RuntimeError("Radar-only benchmark ??? ?? ????.")


---
### FMCW ?? ? ?? 6/7 ? **Radar-only ???? ??**

???? **LiDAR ?? ??? ? ?? ???? ??**, ?????? ?? feature / ???? ??? ?????.

- **3-scan baseline**: ?? ???
- **5-scan rich**: RCS?????? feature ??
- **5-scan temporal**: ?? ?? ????? ??

LiDAR? **validation???** ??, ??? ?? peak? ?? LiDAR cluster? ??? ????? ?????.


## 6. ?? ??? BEV ?? (TOP-2 Radar-only ??) + ??? ?? + LiDAR DBSCAN

**B (????)** ?? ????? ???? ?? 2? **Radar-only** ??? peak? ??, LiDAR DBSCAN? **?? ?? ???**?? ???.


In [ ]:
top_radar_views = []
for item in radar_experiments[:2]:
    if radar_pts.size > 0:
        view = bev.predict_radar_peaks(
            item["model"],
            radar_pts,
            bev_cfg,
            device,
            feature_channels=item["feature_channels"],
            heat_thresh=item["heat_thresh"],
        )
    else:
        view = {
            "prob": None,
            "peaks_by_class": [[] for _ in range(3)],
            "merged_peaks": [],
        }
    view["name"] = item["name"]
    view["radar_mode"] = item["radar_mode"]
    view["feature_channels"] = item["feature_channels"]
    view["metrics"] = item["metrics"]
    view["lidar_validation"] = item["lidar_validation"]
    view["heat_thresh"] = item["heat_thresh"]
    top_radar_views.append(view)

best_radar_view = top_radar_views[0] if top_radar_views else None

# ??? XY (?? ?? ??, ??? ?? ? ?? ??)
radar_xy = []
if FUSION and FUSION.get("radarDetections"):
    for d in FUSION["radarDetections"]:
        c = d.get("centroidM") or d.get("centroid_m")
        if c and len(c) >= 2:
            radar_xy.append((float(c[0]), float(c[1])))
elif radar_pts.size > 0:
    xy = radar_pts[:, :2]
    m = (
        (xy[:, 0] >= bev_cfg.x_min)
        & (xy[:, 0] <= bev_cfg.x_max)
        & (np.abs(xy[:, 1]) <= max(abs(bev_cfg.y_min), abs(bev_cfg.y_max)))
    )
    sub = xy[m]
    if len(sub) > 20:
        db = DBSCAN(eps=1.2, min_samples=4).fit(sub)
        for lab in set(db.labels_):
            if lab < 0:
                continue
            radar_xy.append(tuple(sub[db.labels_ == lab].mean(axis=0).tolist()))

# LiDAR DBSCAN -> BEV ??? ??
lidar_boxes = []
xyz = lidar_pts[:, :3]
m = (
    (xyz[:, 0] >= bev_cfg.x_min)
    & (xyz[:, 0] <= bev_cfg.x_max)
    & (xyz[:, 1] >= bev_cfg.y_min)
    & (xyz[:, 1] <= bev_cfg.y_max)
)
sub = xyz[m]
if len(sub) > 30:
    db2 = DBSCAN(eps=0.9, min_samples=10).fit(sub[:, :2])
    for lab in set(db2.labels_):
        if lab < 0:
            continue
        blk = sub[db2.labels_ == lab]
        xm, ym = blk[:, 0].min(), blk[:, 1].min()
        xM, yM = blk[:, 0].max(), blk[:, 1].max()
        lidar_boxes.append(((xm, ym), (xM, yM)))

print("radar centers (plot):", len(radar_xy), "| lidar cluster boxes:", len(lidar_boxes))
for view in top_radar_views:
    counts = [len(v) for v in view["peaks_by_class"]]
    print(
        f"  {view['name']} peaks V/P/C: {counts[0]}/{counts[1]}/{counts[2]} | "
        f"gt_f1={view['metrics']['micro_f1']:.4f} | lidar_f1={view['lidar_validation']['cluster_f1']:.4f}"
    )


---
### FMCW ?? ? ?? 7/7 ? **?? ? ?? ? ?? Radar-only ?? + LiDAR ???**

? ?? **???-only? ??? ?? ???**? ?? ??? ?? peak? ??? ????, LiDAR? **?? ???? ?? ???**??? ????.


## 7. 통합 디스플레이 (2×4) — **FMCW 콘솔 “한 화면” 요약**

아래 코드를 실행하면 **한 스냅샷**에 모든 레이어가 올라갑니다.

| 칸 | 의미 (FMCW 관점) |
|----|------------------|
| ① 원본 카메라 | EO/IR 참조 |
| ② YOLO | 식별 패널 |
| ③ BEV LiDAR 배경 + 레이더 X | 공통 지도 위 **레이더 후보** |
| ④ LiDAR BEV CNN + GT | (참고) 상한 |
| ⑤ 레이더만 BEV CNN + GT | **배포에 가까운 후보** |
| ⑥ LiDAR DBSCAN | (참고) 규칙 기반 |

**그림이 이전과 같으면** 커널 **Restart** 후 위에서부터 실행하세요. BEV 학습 셀에서 `model_lidar`·`model_radar`가 정의됩니다.



In [ ]:
def plot_bev_background(ax, pts_xyz, cfg, max_pts=12000):
    m = (
        (pts_xyz[:, 0] >= cfg.x_min)
        & (pts_xyz[:, 0] <= cfg.x_max)
        & (pts_xyz[:, 1] >= cfg.y_min)
        & (pts_xyz[:, 1] <= cfg.y_max)
    )
    p = pts_xyz[m, :2]
    if p.shape[0] > max_pts:
        rng = np.random.default_rng(0)
        p = p[rng.choice(p.shape[0], max_pts, replace=False)]
    ax.scatter(p[:, 1], p[:, 0], s=0.15, c="0.6", alpha=0.35, rasterized=True)
    ax.set_xlim(cfg.y_max, cfg.y_min)
    ax.set_ylim(cfg.x_min, cfg.x_max)
    ax.set_aspect("equal")
    ax.set_xlabel("Y (velo, m)")
    ax.set_ylabel("X (velo, m)")


def _bev_panel_tag(ax, tag: str) -> None:
    ax.text(
        0.02,
        0.98,
        tag,
        transform=ax.transAxes,
        fontsize=10,
        fontweight="bold",
        color="white",
        va="top",
        ha="left",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.5),
        zorder=20,
    )


def _counts_text(view):
    if view is None:
        return "n/a"
    return "/".join(str(len(v)) for v in view["peaks_by_class"])


top1 = top_radar_views[0] if len(top_radar_views) > 0 else None
top2 = top_radar_views[1] if len(top_radar_views) > 1 else None

fig = plt.figure(figsize=(19, 9.2))
gs = fig.add_gridspec(2, 4, hspace=0.32, wspace=0.24)

# ----- row 0 -----
ax = fig.add_subplot(gs[0, 0])
if img_raw is not None:
    ax.imshow(img_raw)
    ax.set_title("? EO/IR ?? (?? ???)")
else:
    ax.text(0.5, 0.5, "no image", ha="center")
ax.axis("off")

ax = fig.add_subplot(gs[0, 1])
if FUSION and FUSION.get("annotatedImageBase64"):
    ax.imshow(vcu.b64_to_pil_image(FUSION["annotatedImageBase64"]))
    ax.set_title("? ?? ?? ? YOLO (?? API)")
elif img_raw is not None:
    ax.imshow(img_raw)
    ax.set_title("? ?? ?? (??? ? ??)")
else:
    ax.text(0.5, 0.5, "YOLO N/A", ha="center")
ax.axis("off")

ax = fig.add_subplot(gs[0, 2:])
ax.axis("off")
lines = [
    "[??] Radar-only ?? + LiDAR validation ????",
    f"frame: {FRAME_ID} | radar view: {RADAR_VIEW_MODE}",
    f"LiDAR pts: {lidar_pts.shape[0]} | Radar pts: {radar_pts.shape[0]}",
    f"GT boxes (label_2): {len(gt_footprints)}",
    f"TOP1 peaks V/P/C: {_counts_text(top1)} | TOP2 peaks V/P/C: {_counts_text(top2)}",
    f"Radar xy (??/????): {len(radar_xy)} | LiDAR DBSCAN boxes: {len(lidar_boxes)}",
]
for row in leaderboard[:3]:
    lines.append(
        f"#{row['rank']} {row['name']} | score={row['selection_score']:.4f} | "
        f"gt_f1={row['micro_f1']:.4f} | lidar_f1={row['lidar_cluster_f1']:.4f}"
    )
if FUSION_ERROR:
    lines.append("AI: " + FUSION_ERROR[:80])
ax.text(0, 1, "
".join(lines), va="top", fontsize=10, family="monospace", transform=ax.transAxes)

# ----- row 1: BEV ?? 4? -----
ax = fig.add_subplot(gs[1, 0])
plot_bev_background(ax, lidar_pts[:, :3], bev_cfg)
for x, y in radar_xy:
    ax.scatter(y, x, c="red", s=36, marker="x", zorder=5)
_bev_panel_tag(ax, "??: LiDAR ?")
ax.set_title("? FMCW ???: BEV ?? + ??? ??(X)")

cols = ["#ff7f0e", "#e377c2", "#17becf"]

ax = fig.add_subplot(gs[1, 1])
plot_bev_background(ax, lidar_pts[:, :3], bev_cfg)
for g in gt_footprints:
    poly = g["polygon_xy"]
    ax.add_patch(Polygon(poly[:, ::-1], closed=True, fill=False, edgecolor="lime", linewidth=1.8))
if top1 is not None:
    for c, pname in enumerate(class_names):
        first = True
        for x, y in top1["peaks_by_class"][c]:
            ax.scatter(y, x, c=cols[c], s=22, alpha=0.9, label=pname if first else "_nolegend_")
            first = False
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="upper right", fontsize=7)
    _bev_panel_tag(ax, f"TOP1 ? {top1['radar_mode']}")
    ax.set_title(f"? {top1['name']} + GT")
else:
    ax.text(0.5, 0.5, "TOP1 ??", ha="center", va="center", transform=ax.transAxes)
    _bev_panel_tag(ax, "TOP1 ??")
    ax.set_title("? Radar-only TOP1")

ax = fig.add_subplot(gs[1, 2])
plot_bev_background(ax, lidar_pts[:, :3], bev_cfg)
for g in gt_footprints:
    poly = g["polygon_xy"]
    ax.add_patch(Polygon(poly[:, ::-1], closed=True, fill=False, edgecolor="lime", linewidth=1.8))
if top2 is not None:
    for c, pname in enumerate(class_names):
        first = True
        for x, y in top2["peaks_by_class"][c]:
            ax.scatter(y, x, c=cols[c], s=22, alpha=0.9, label=pname if first else "_nolegend_")
            first = False
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="upper right", fontsize=7)
    _bev_panel_tag(ax, f"TOP2 ? {top2['radar_mode']}")
    ax.set_title(f"? {top2['name']} + GT")
else:
    ax.text(0.5, 0.5, "TOP2 ??", ha="center", va="center", transform=ax.transAxes)
    _bev_panel_tag(ax, "TOP2 ??")
    ax.set_title("? Radar-only TOP2")

ax = fig.add_subplot(gs[1, 3])
plot_bev_background(ax, lidar_pts[:, :3], bev_cfg)
for (xm, ym), (xM, yM) in lidar_boxes:
    rect = mpatches.Rectangle(
        (ym, xm), yM - ym, xM - xm, fill=False, edgecolor="cyan", linewidth=1.5
    )
    ax.add_patch(rect)
_bev_panel_tag(ax, "??: LiDAR DBSCAN")
ax.set_title("? LiDAR DBSCAN ?? (???)")

fig.suptitle(
    f"[FMCW ?? ?? 2?4] Radar-only ??, LiDAR ?? ? frame {FRAME_ID}",
    fontsize=13,
    fontweight="bold",
)
fig.text(
    0.5,
    0.005,
    "?? ?: ???+???X ?TOP1 Radar-only ?TOP2 Radar-only ?LiDAR DBSCAN(??)",
    ha="center",
    fontsize=10,
    color="navy",
)
plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()


---

### 참고 (용어·운영 정리 · FMCW 관점)

- **`strong_validated` / `radar_only` 등 LiDAR 검증 태그**는 **운영 화면의 핵심 구분**으로 두기보다, **오프라인 분석**에 두고, 운영에서는 **`high_confidence` / `medium_confidence` / `low_confidence`**처럼 **레이더 특징 기반** 등급으로 치환하는 것을 권장합니다.  
- **학습 CNN** 두 종류는 동일 히트맵 타깃에 대해 **입력만** LiDAR BEV vs 레이더 BEV로 다릅니다 (3D 박스 직접 회귀 아님).  
- **GT 초록 다각형**은 KITTI `label_2` 바닥면을 velodyne으로 변환한 것입니다 (**B 구간 평가 기준**).  
- **YOLO**는 이미지 좌표계라 BEV와 직접 정합되지 않습니다 — **나란히 정성 비교**용입니다.  
- 전체 mAP·정밀한 레이더–카메라 정합은 별도 파이프라인이 필요합니다.

